# 04 — Results analysis

Aggregate metrics across runs (training seeds), compare systems with the
**paired bootstrap**, and analyze errors and lexical overlap of imported
prediction exports.

**Two different sources of uncertainty (do not mix):**
- *Seed variation*: statistics across training runs of the same configuration
  (`aggregate_seeds`) — run-level variation.
- *Test-set sampling uncertainty*: sentence-level bootstrap CIs per run
  (`bootstrap_ci`) — resampling the same test sentences.

Repeated seeds are NOT independent test datasets.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))
print("repo root:", repo_root)

In [ ]:
from axner.evaluation.aggregation import aggregate_run_directories

outputs_dir = repo_root / "outputs"
run_dirs = [
    run_dir
    for exp_dir in sorted(p for p in outputs_dir.iterdir() if p.is_dir())
    for run_dir in sorted(p for p in exp_dir.iterdir() if p.is_dir())
]
print(len(run_dirs), "run directories found")
aggregated = aggregate_run_directories(run_dirs)
aggregated

In [ ]:
import pandas as pd

rows = []
for key, entry in aggregated.items():
    f1 = entry["aggregation"]["metrics"].get("f1", {})
    rows.append({
        "experiment": entry["experiment_id"],
        "target": entry["target_language"],
        "variant": str(entry["variant_tags"]),
        "f1_mean": f1.get("mean"),
        "f1_std": f1.get("std"),
        "n_seeds": f1.get("n_values"),
    })
pd.DataFrame(rows)

## Per-run metrics and bootstrap confidence intervals

Loads `metrics/test_metrics_<lang>.json` and `metrics/bootstrap_<lang>.json`
from one run directory (e.g. an imported Colab/Kaggle bundle).

In [ ]:
from axner.artifacts.manager import load_json

if run_dirs:
    run_dir = run_dirs[-1]
    manifest = load_json(run_dir / "manifest.json")
    print("run:", manifest["experiment_id"], manifest["run_id"])
    for path in sorted((run_dir / "metrics").glob("test_metrics_*.json")):
        metrics = load_json(path)
        print(path.name, "f1=", metrics["f1"], "P=", metrics["precision"], "R=", metrics["recall"])
    for path in sorted((run_dir / "metrics").glob("bootstrap_*.json")):
        b = load_json(path)
        print(path.name, f"95% CI [{b['ci_low']:.3f}, {b['ci_high']:.3f}]")

## Paired bootstrap between two runs (same target test set)

Valid only when both runs evaluated the SAME test sentences (same language,
same full test split).

In [ ]:
from axner.evaluation.bootstrap import paired_bootstrap

def load_pred_tags(run_dir, lang):
    from axner.artifacts.manager import load_jsonl
    return [r["pred_tags"] for r in load_jsonl(run_dir / "predictions" / f"test_{lang}_sentence_level.jsonl")]

def load_gold_tags(run_dir, lang):
    from axner.artifacts.manager import load_jsonl
    return [r["gold_tags"] for r in load_jsonl(run_dir / "predictions" / f"test_{lang}_sentence_level.jsonl")]

if len(run_dirs) >= 2:
    a, b = run_dirs[-1], run_dirs[-2]
    lang = "jv"
    result = paired_bootstrap(load_gold_tags(a, lang), load_pred_tags(a, lang),
                              load_pred_tags(b, lang), n_resamples=1000, seed=42)
    result

## Seen vs unseen entity performance (lexical overlap)

Join a run's entity-level predictions with the lexical-overlap classification
of each test entity against the source training inventory.

In [ ]:
import pandas as pd

from axner.analysis.lexical_overlap import build_inventory, classify_mentions
from axner.analysis.normalization import normalize_surface
from axner.constants import LABEL_NAMES
from axner.data.wikiann import load_wikiann

if run_dirs:
    # pick the run's init source for the inventory (edit for your run)
    source_lang = "id"
    target_lang = "jv"
    source_train = load_wikiann(source_lang, "train", cache_dir=str(repo_root / "data" / "hf_cache"))
    inventory = build_inventory(source_train, LABEL_NAMES)

    from axner.artifacts.manager import load_jsonl

    entities = load_jsonl(run_dir / "predictions" / f"test_{target_lang}_entity_level.jsonl")
    inv_keys = {(s, t) for t, surfaces in inventory.items() for s in surfaces}
    df = pd.DataFrame(entities)
    df["seen"] = [
        (normalize_surface(e["surface"]), e["type"]) in inv_keys
        for e in entities
    ]
    df.groupby(["seen", "kind"]).size().unstack(fill_value=0)

**Reading the table:** rows group entities by whether the target-test
surface+type appeared in the source training data (`seen`); columns are
tp/fp/fn. Comparing F1 on seen vs unseen subsets quantifies how much of the
transfer performance is lexical memorization vs structural generalization.
For the full similar/unseen taxonomy use
`axner.analysis.lexical_overlap.overlap_report` (notebook 02).